# SimCLR to YOLO26 Football Detection

This tutorial transfers a trained SimCLR backbone into YOLO26, fine-tunes it for football object detection, evaluates the best detector on labelled test images, and analyzes a football video with detection and tracking.

**Inputs**

- SimCLR checkpoint: /kaggle/input/notebooks/rifat963/cse445-simclr-football-t4x2-tutorial/simclr_football/best_ssl.pt
- Football detection dataset from iasadpanwhar/football-player-detection-yolov8
- Video: /kaggle/input/datasets/iasadpanwhar/football-player-detection-yolov8/video.mp4

**Learning goals**

- Understand which part of a SimCLR checkpoint transfers to detection
- Verify that the SSL backbone matches YOLO26
- Warm up a randomly initialized detection head
- Fine-tune the complete detector on two T4 GPUs
- Export precision, recall, F1, AP, and mAP metrics
- Run video detection, tracking, and performance analysis


## Transfer pipeline

The SSL checkpoint contains the online encoder, projection head, optimizer, scheduler, and training state. Object detection needs only the online encoder.

SimCLR best_ssl.pt → online encoder → YOLO26 backbone → head warm-up → full fine-tuning → labelled evaluation → video tracking

The projection head is excluded because it was created for the contrastive objective. The YOLO detection neck and head are initialized by the YOLO26 architecture and learned from football bounding-box labels.


## Notebook roadmap

1. Configure Kaggle and install the library
2. Validate the checkpoint, dataset, and video
3. Create an absolute-path dataset YAML
4. Transfer the SimCLR encoder into YOLO26
5. Warm up the detection head
6. Fine-tune the complete detector
7. Review training curves
8. Inspect predictions
9. Evaluate the labelled test set
10. Analyze and display the video
11. Export the results


## 1. Configure Kaggle

Select GPU T4 x2, enable Internet access, and attach both the football dataset and the completed SimCLR notebook as inputs. The notebook reads the mounted paths directly and does not use the Kaggle competition API.


In [ ]:
%pip install -q --no-cache-dir --force-reinstall --no-deps "git+https://github.com/rifat963/ssl-detection-lab.git@main"


In [ ]:
from pathlib import Path
import json
import random
import shutil
import sys
from importlib.metadata import version as installed_version

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from IPython.display import Markdown, Video, display
from PIL import Image
from packaging.version import Version
from ultralytics import YOLO

INSTALLED_SSLDET_VERSION = installed_version("ssl-detection-lab")
assert Version(INSTALLED_SSLDET_VERSION) >= Version("0.6.0"), (
    f"Installed ssl-detection-lab {INSTALLED_SSLDET_VERSION}; version 0.6.0 or newer is required. "
    "Publish the current library to GitHub main, rerun the installation cell, and restart the session."
)

for module_name in list(sys.modules):
    if module_name == "ssldet" or module_name.startswith("ssldet."):
        sys.modules.pop(module_name)

import ssldet
from ssldet import (
    EvaluationConfig,
    VideoAnalysisConfig,
    analyze_video,
    evaluate,
    transfer_ssl_backbone_to_yolo,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision("high")
sns.set_theme(style="whitegrid", context="notebook")

assert Version(ssldet.__version__) >= Version("0.6.0")
assert torch.cuda.is_available(), "Select a GPU accelerator before continuing."

GPU_COUNT = torch.cuda.device_count()
DEVICE_IDS = list(range(GPU_COUNT))
GPU_NAMES = [torch.cuda.get_device_name(index) for index in DEVICE_IDS]

pd.Series({
    "ssl-detection-lab": ssldet.__version__,
    "PyTorch": torch.__version__,
    "Ultralytics": __import__("ultralytics").__version__,
    "CUDA": torch.version.cuda,
    "GPU count": GPU_COUNT,
    "GPUs": ", ".join(GPU_NAMES),
})


The expected environment has two NVIDIA T4 GPUs. Ultralytics will launch distributed training automatically when both device IDs are supplied.


## 2. Validate the inputs


In [ ]:
SSL_CANDIDATES = [
    Path("/kaggle/input/notebooks/rifat963/cse445-simclr-football-t4x2-tutorial/simclr_football/best_ssl.pt"),
    Path("/kaggle/input/cse445-simclr-football-t4x2-tutorial/simclr_football/best_ssl.pt"),
]
DATASET_CANDIDATES = [
    Path("/kaggle/input/datasets/iasadpanwhar/football-player-detection-yolov8/football_players_detection/football_players_detection"),
    Path("/kaggle/input/football-player-detection-yolov8/football_players_detection/football_players_detection"),
]
VIDEO_CANDIDATES = [
    Path("/kaggle/input/datasets/iasadpanwhar/football-player-detection-yolov8/video.mp4"),
    Path("/kaggle/input/football-player-detection-yolov8/video.mp4"),
]

SSL_CHECKPOINT = next((path for path in SSL_CANDIDATES if path.is_file()), None)
DATASET_ROOT = next((path for path in DATASET_CANDIDATES if path.is_dir()), None)
VIDEO_FILE = next((path for path in VIDEO_CANDIDATES if path.is_file()), None)

if SSL_CHECKPOINT is None:
    raise FileNotFoundError("Attach the completed SimCLR notebook output containing best_ssl.pt")
if DATASET_ROOT is None:
    raise FileNotFoundError("Attach the football-player-detection-yolov8 dataset")
if VIDEO_FILE is None:
    raise FileNotFoundError("The football video.mp4 file was not found in the dataset input")

SPLIT_PATHS = {
    split: {
        "images": DATASET_ROOT / split / "images",
        "labels": DATASET_ROOT / split / "labels",
    }
    for split in ("train", "valid", "test")
}

pd.Series({
    "SSL checkpoint": str(SSL_CHECKPOINT),
    "SSL checkpoint MB": round(SSL_CHECKPOINT.stat().st_size / 1024 ** 2, 2),
    "dataset root": str(DATASET_ROOT),
    "video": str(VIDEO_FILE),
    "video MB": round(VIDEO_FILE.stat().st_size / 1024 ** 2, 2),
})


In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def image_count(directory):
    return sum(
        path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
        for path in directory.rglob("*")
    )

split_summary = []
for split, paths in SPLIT_PATHS.items():
    split_summary.append({
        "split": split,
        "images": image_count(paths["images"]),
        "labels": len(list(paths["labels"].glob("*.txt"))),
    })

pd.DataFrame(split_summary).set_index("split")


The SimCLR checkpoint metadata records the pretraining method and source YOLO architecture. The transfer must use the same backbone architecture; otherwise tensor shapes or key coverage will fail.


In [ ]:
ssl_state = torch.load(SSL_CHECKPOINT, map_location="cpu", weights_only=True)
ssl_config = dict(ssl_state.get("config", {}))
SSL_METHOD = str(ssl_config.get("method", "unknown"))
SSL_SOURCE_MODEL = str(ssl_config.get("yolo_model", "yolo26n.yaml"))
METHOD_KEYS = list(ssl_state.get("method", {}))
ONLINE_ENCODER_KEYS = [key for key in METHOD_KEYS if key.startswith("online_encoder.")]

assert SSL_METHOD == "simclr", f"Expected SimCLR, found {SSL_METHOD}"
assert "yolo26" in SSL_SOURCE_MODEL.lower(), f"Expected YOLO26, found {SSL_SOURCE_MODEL}"
assert ONLINE_ENCODER_KEYS, "No online_encoder weights were found"

pd.Series({
    "SSL method": SSL_METHOD,
    "source model": SSL_SOURCE_MODEL,
    "completed epochs": ssl_state.get("epoch"),
    "best SSL loss": ssl_state.get("best_loss"),
    "method state entries": len(METHOD_KEYS),
    "online encoder entries": len(ONLINE_ENCODER_KEYS),
})


## 3. Create the detection dataset YAML

Absolute image paths avoid path-resolution errors inside Kaggle and Ultralytics distributed workers.


In [ ]:
source_yaml_candidates = sorted(DATASET_ROOT.parent.rglob("data.yaml"))
source_yaml = source_yaml_candidates[0] if source_yaml_candidates else None
source_metadata = yaml.safe_load(source_yaml.read_text()) if source_yaml else {}
CLASS_NAMES = source_metadata.get("names", ["ball", "goalkeeper", "player", "referee"])

DATA_YAML = Path("/kaggle/working/football_detection.yaml")
dataset_definition = {
    "train": str(SPLIT_PATHS["train"]["images"]),
    "val": str(SPLIT_PATHS["valid"]["images"]),
    "test": str(SPLIT_PATHS["test"]["images"]),
    "names": CLASS_NAMES,
}
DATA_YAML.write_text(yaml.safe_dump(dataset_definition, sort_keys=False))

print(DATA_YAML.read_text())


## 4. Transfer SimCLR into YOLO26

The transfer utility loads online_encoder entries only, strips their SSL prefix, checks them against the YOLO26 backbone, and writes a detector-compatible checkpoint. Projection-head and optimizer states are deliberately excluded.


In [ ]:
PROJECT_DIR = Path("/kaggle/working/yolo26_simclr_downstream")
INITIALIZED_DETECTOR = PROJECT_DIR / "simclr_initialized_yolo26n.pt"

transfer_result = transfer_ssl_backbone_to_yolo(
    SSL_CHECKPOINT,
    INITIALIZED_DETECTOR,
    yolo_model=SSL_SOURCE_MODEL,
    minimum_coverage=0.95,
)

pd.Series({
    "SSL method": transfer_result.ssl_method,
    "source architecture": transfer_result.source_model,
    "encoder prefix": transfer_result.encoder_prefix,
    "loaded keys": transfer_result.loaded_keys,
    "total backbone keys": transfer_result.total_backbone_keys,
    "coverage": f"{transfer_result.coverage:.2%}",
    "missing keys": len(transfer_result.missing_keys),
    "unexpected keys": len(transfer_result.unexpected_keys),
    "detector checkpoint": str(transfer_result.detector_checkpoint),
})


Coverage should be 100 percent when the checkpoint and installed Ultralytics architecture match exactly. Training should not continue when coverage is below the required threshold.


In [ ]:
transfer_report = json.loads(transfer_result.report_json.read_text())
display(pd.DataFrame({
    "missing key": pd.Series(transfer_report["missing_keys"], dtype="object"),
    "unexpected key": pd.Series(transfer_report["unexpected_keys"], dtype="object"),
}).fillna(""))


## 5. Configure downstream training

The first stage freezes the transferred backbone for three epochs while the randomly initialized detection layers learn a stable starting point. The second stage reloads the best warm-up checkpoint and fine-tunes every layer for 30 epochs.

AMP remains enabled. The batch value is the global Ultralytics batch and is divided across the two GPUs.


In [ ]:
IMAGE_SIZE = 640
GLOBAL_BATCH_SIZE = 32
HEAD_WARMUP_EPOCHS = 3
FINETUNE_EPOCHS = 30
WORKERS = 2

training_settings = pd.Series({
    "image size": IMAGE_SIZE,
    "global batch": GLOBAL_BATCH_SIZE,
    "head warm-up epochs": HEAD_WARMUP_EPOCHS,
    "full fine-tuning epochs": FINETUNE_EPOCHS,
    "devices": DEVICE_IDS,
    "AMP": True,
})
training_settings


## 6. Warm up the detection head


In [ ]:
warmup_model = YOLO(str(INITIALIZED_DETECTOR))
BACKBONE_LAYERS = len(warmup_model.model.yaml["backbone"])

warmup_model.train(
    data=str(DATA_YAML),
    epochs=HEAD_WARMUP_EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=GLOBAL_BATCH_SIZE,
    device=DEVICE_IDS,
    workers=WORKERS,
    freeze=BACKBONE_LAYERS,
    optimizer="AdamW",
    lr0=1e-3,
    lrf=0.1,
    weight_decay=5e-4,
    warmup_epochs=1,
    amp=True,
    seed=SEED,
    deterministic=True,
    project=str(PROJECT_DIR),
    name="head_warmup",
    exist_ok=True,
    plots=True,
    verbose=True,
)

WARMUP_BEST = PROJECT_DIR / "head_warmup" / "weights" / "best.pt"
assert WARMUP_BEST.is_file()
WARMUP_BEST


## 7. Fine-tune the complete detector

Every layer is now trainable. The lower learning rate protects the transferred representation while allowing the backbone to specialize for football detection.


In [ ]:
finetune_model = YOLO(str(WARMUP_BEST))
finetune_model.train(
    data=str(DATA_YAML),
    epochs=FINETUNE_EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=GLOBAL_BATCH_SIZE,
    device=DEVICE_IDS,
    workers=WORKERS,
    freeze=0,
    optimizer="AdamW",
    lr0=3e-4,
    lrf=0.01,
    weight_decay=5e-4,
    warmup_epochs=1,
    cos_lr=True,
    close_mosaic=10,
    patience=10,
    amp=True,
    seed=SEED,
    deterministic=True,
    project=str(PROJECT_DIR),
    name="full_finetune",
    exist_ok=True,
    plots=True,
    save_period=5,
    verbose=True,
)

BEST_DETECTOR = PROJECT_DIR / "full_finetune" / "weights" / "best.pt"
LAST_DETECTOR = PROJECT_DIR / "full_finetune" / "weights" / "last.pt"
assert BEST_DETECTOR.is_file()

pd.Series({
    "best detector": str(BEST_DETECTOR),
    "last detector": str(LAST_DETECTOR),
    "best checkpoint MB": round(BEST_DETECTOR.stat().st_size / 1024 ** 2, 2),
})


## 8. Review the downstream training curves


In [ ]:
training_history = pd.read_csv(PROJECT_DIR / "full_finetune" / "results.csv")
training_history.columns = training_history.columns.str.strip()
display(training_history.tail().round(5))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
curve_specs = [
    ("train/box_loss", "Training box loss"),
    ("val/box_loss", "Validation box loss"),
    ("metrics/mAP50(B)", "Validation mAP50"),
    ("metrics/mAP50-95(B)", "Validation mAP50-95"),
]
for axis, (column, title) in zip(axes.flat, curve_specs):
    if column in training_history:
        sns.lineplot(data=training_history, x="epoch", y=column, linewidth=2.4, ax=axis)
    axis.set_title(title)
    axis.xaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout()
plt.show()


Lower localization losses and rising mAP indicate that the transferred representation is adapting to the downstream labels. The test set remains unused until the final evaluation.


## 9. Inspect test-image predictions


In [ ]:
test_images = sorted(
    path for path in SPLIT_PATHS["test"]["images"].rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
)
prediction_paths = random.Random(SEED).sample(test_images, min(6, len(test_images)))
prediction_model = YOLO(str(BEST_DETECTOR))
prediction_results = prediction_model.predict(
    source=[str(path) for path in prediction_paths],
    imgsz=IMAGE_SIZE,
    conf=0.25,
    iou=0.7,
    device=0,
    half=True,
    verbose=False,
)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for axis in axes.flat:
    axis.axis("off")
for axis, result in zip(axes.flat, prediction_results):
    axis.imshow(result.plot()[..., ::-1])
    axis.set_title(Path(result.path).name)
plt.tight_layout()
plt.show()


## 10. Evaluate the labelled test set

This stage exports headline metrics, per-class precision and recall, F1, AP50, AP50-95, speed measurements, curves, and the confusion matrix.


In [ ]:
EVALUATION_DIR = PROJECT_DIR / "test_evaluation"
evaluation_result = evaluate(EvaluationConfig(
    model_name="yolo26n",
    weights_file=str(BEST_DETECTOR),
    data=str(DATA_YAML),
    output_dir=str(EVALUATION_DIR),
    split="test",
    image_size=IMAGE_SIZE,
    batch_size=GLOBAL_BATCH_SIZE,
    confidence=0.001,
    iou=0.7,
    max_detections=300,
    device=0,
    workers=WORKERS,
    half=True,
    plots=True,
    save_json=True,
))

evaluation_result


In [ ]:
evaluation_report = json.loads(evaluation_result.metrics_json.read_text())
headline_metrics = evaluation_report["headline_metrics"]
display(pd.Series(headline_metrics, name="test value").to_frame().round(5))

if evaluation_result.per_class_csv is not None:
    per_class_metrics = pd.read_csv(evaluation_result.per_class_csv)
    display(per_class_metrics.round(5))


In [ ]:
confusion_images = sorted(EVALUATION_DIR.rglob("confusion_matrix_normalized.png"))
if confusion_images:
    with Image.open(confusion_images[0]) as image:
        display(image.copy())


The labelled test set supports accuracy claims such as precision, recall, AP, and mAP. These metrics must not be calculated from the unlabelled video unless ground-truth video annotations are available.


## 11. Analyze the football video

The video module performs streaming inference with BoT-SORT tracking. It exports frame-level rows, detection-level rows, class distributions, confidence statistics, box occupancy, latency, throughput, track counts, and an annotated video.


In [ ]:
VIDEO_OUTPUT_DIR = PROJECT_DIR / "video_analysis"
video_result = analyze_video(VideoAnalysisConfig(
    video_source=str(VIDEO_FILE),
    model_name="yolo26n",
    weights_file=str(BEST_DETECTOR),
    output_dir=str(VIDEO_OUTPUT_DIR),
    confidence=0.25,
    iou=0.7,
    image_size=IMAGE_SIZE,
    max_detections=300,
    device=0,
    tracker="botsort.yaml",
    video_stride=1,
    max_frames=None,
    save_annotated=True,
    save_txt=False,
    save_confidence=False,
))

video_result


In [ ]:
video_report = json.loads(video_result.report_json.read_text())
video_metrics = video_report["video_metrics"]
confidence_stats = video_metrics["confidence"]
latency_stats = video_metrics["latency_ms"]

pd.Series({
    "frames processed": video_metrics["frames_processed"],
    "total detections": video_metrics["total_detections"],
    "frames with detections": video_metrics["frames_with_detections"],
    "detection coverage": video_metrics["detection_frame_coverage"],
    "mean confidence": confidence_stats["mean"],
    "mean inference latency ms": latency_stats["mean"],
    "processing FPS": video_metrics["processing_fps"],
    "unique tracks": video_metrics["tracking"]["unique_tracks"],
}).to_frame("value")


In [ ]:
per_class_video = pd.DataFrame(video_report["per_class"])
if not per_class_video.empty:
    per_class_video["mean_confidence"] = per_class_video["confidence"].map(lambda value: value["mean"])
    display(per_class_video[[
        "class_id",
        "class_name",
        "detections",
        "frames_present",
        "frame_coverage",
        "unique_tracks",
        "mean_confidence",
    ]].round(4))


In [ ]:
frame_metrics = pd.read_csv(video_result.frames_csv)
detection_metrics = pd.read_csv(video_result.detections_csv)

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
sns.lineplot(data=frame_metrics, x="frame", y="detections", linewidth=1.4, ax=axes[0])
sns.lineplot(data=frame_metrics, x="frame", y="inference_ms", linewidth=1.2, ax=axes[1])
axes[0].set_title("Detections per analyzed frame")
axes[0].set_ylabel("Detections")
axes[1].set_title("Inference latency per frame")
axes[1].set_ylabel("Milliseconds")
plt.tight_layout()
plt.show()


In [ ]:
display(Markdown(video_result.outcome_markdown.read_text()))


## 12. Display the annotated video


In [ ]:
if video_result.annotated_media:
    display(Video(str(video_result.annotated_media[0]), embed=True, width=960))
else:
    print("No annotated video file was discovered in the output directory")


## 13. Export the complete downstream run


In [ ]:
ARCHIVE_FILE = Path(shutil.make_archive(
    "/kaggle/working/yolo26_simclr_downstream_results",
    "zip",
    root_dir=PROJECT_DIR,
))

pd.Series({
    "best detector": str(BEST_DETECTOR),
    "evaluation metrics": str(evaluation_result.metrics_json),
    "video report": str(video_result.report_json),
    "annotated media": ", ".join(str(path) for path in video_result.annotated_media),
    "results archive": str(ARCHIVE_FILE),
    "archive MB": round(ARCHIVE_FILE.stat().st_size / 1024 ** 2, 2),
})


## Exercise: measure the value of SimCLR

Train the same YOLO26 architecture from random initialization using the identical split, seed, augmentations, epoch budget, image size, and batch size. Evaluate it on the same test set. The only changed variable should be the backbone initialization.


In [ ]:
metric_keys = [
    "metrics/precision(B)",
    "metrics/recall(B)",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)",
]
comparison = pd.DataFrame([
    {
        "initialization": "SimCLR",
        **{key: headline_metrics.get(key) for key in metric_keys},
    },
    {
        "initialization": "random YOLO26",
        **{key: np.nan for key in metric_keys},
    },
]).set_index("initialization")
comparison


Fill the random-initialization row with its test metrics, then calculate the absolute mAP improvement produced by SimCLR. One run is a demonstration; repeated seeds are needed for a reliable scientific comparison.


## Practical checks

- A low transfer coverage means the YOLO26 scale or Ultralytics architecture differs from pretraining.
- Reduce the global batch from 32 to 16 if distributed training runs out of memory.
- The detector head must be trained with labels because SimCLR pretraining contains no detection head supervision.
- Test metrics are valid only when the test split was not used for model selection.
- Video counts and tracks are predictions, not ground truth.
- MOTA, IDF1, and HOTA require ground-truth track identities and are not reported for this unlabelled video.
- Ultralytics is available under AGPL-3.0 and an Enterprise License. Review https://www.ultralytics.com/license for the intended use.
